# Evaluate dataset1 retrieval models

Compare the base and fine-tuned BGE-M3 models, Jina Embeddings v3, and Snowflake Arctic Embed L v2.0 on `dataset1.jsonl`.

## Setup

Set `FINE_TUNED_MODEL_PATH` to the fine-tuned BGE-M3 directory or Hugging Face model ID. The notebook uses dense cosine retrieval and releases GPU memory between models.

In [ ]:
import csv
import gc
import html
import json
import re
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
from FlagEmbedding import BGEM3FlagModel
from sentence_transformers import SentenceTransformer

def find_project_root(start_path):
    for candidate in (start_path, *start_path.parents):
        if (candidate / 'data').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('Could not find the project root. Run this notebook from inside the repository.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
TEST_INPUT_PATH = PROJECT_ROOT / 'dataset1.jsonl'
REPORT_DIR = PROJECT_ROOT / 'reports/evaluations/dataset1_model_comparison'

BASE_MODEL_NAME = 'BAAI/bge-m3'
FINE_TUNED_MODEL_PATH = PROJECT_ROOT / 'outputs/bge-m3-dense'  # Change if your model is elsewhere.
JINA_MODEL_NAME = 'jinaai/jina-embeddings-v3'
SNOWFLAKE_MODEL_NAME = 'Snowflake/snowflake-arctic-embed-l-v2.0'

BGE_ENCODE_BATCH_SIZE = 16
COMPARISON_ENCODE_BATCH_SIZE = 2
SEARCH_BATCH_SIZE = 64
RETRIEVED_TOP_K = 5
EVALUATION_DEVICES = ['cuda:0'] if torch.cuda.is_available() else ['cpu']
EVALUATION_DEVICE = EVALUATION_DEVICES[0]
USE_FP16 = torch.cuda.is_available()

print(f'Test input: {TEST_INPUT_PATH}')
print(f'Reports: {REPORT_DIR}')
print(f'Device: {EVALUATION_DEVICE}; FP16: {USE_FP16}')


## 1. Load the fixed dataset1 test set

`dataset1.jsonl` contains `id`, `question`, and `answer`. Duplicate answers are retained only once in the corpus, exactly as in the Porseman comparison. `id` is not unique in this dataset, so stable row IDs are generated for report files.

In [ ]:
REQUIRED_FIELDS = {'id', 'question', 'answer'}
if not TEST_INPUT_PATH.is_file():
    raise FileNotFoundError(f'Dataset file was not found: {TEST_INPUT_PATH}')

raw_test_rows = []
with TEST_INPUT_PATH.open(encoding='utf-8') as source:
    for line_number, line in enumerate(source, 1):
        if not line.strip():
            continue
        row = json.loads(line)
        missing_fields = REQUIRED_FIELDS - set(row)
        if missing_fields:
            raise ValueError(f'Line {line_number} is missing fields: {sorted(missing_fields)}')
        row = {field: str(row[field]).strip() for field in REQUIRED_FIELDS}
        row['row_id'] = f'dataset1_{len(raw_test_rows):06d}'
        raw_test_rows.append(row)

invalid_rows = [row for row in raw_test_rows if not row['id'] or not row['question'] or not row['answer']]
if invalid_rows:
    raise ValueError(f'Dataset has {len(invalid_rows):,} row(s) with an empty required field.')

seen_answers = set()
test_rows, duplicate_answer_rows = [], []
for row in raw_test_rows:
    if row['answer'] in seen_answers:
        duplicate_answer_rows.append(row)
    else:
        seen_answers.add(row['answer'])
        test_rows.append(row)

test_row_ids = [row['row_id'] for row in test_rows]
test_source_ids = [row['id'] for row in test_rows]
test_queries = [row['question'] for row in test_rows]
corpus_documents = [row['answer'] for row in test_rows]
relevant_document_indices = np.arange(len(test_rows))

print(f'Rows read: {len(raw_test_rows):,}')
print(f'Rows removed for duplicate answers: {len(duplicate_answer_rows):,}')
print(f'Evaluation queries and corpus documents: {len(test_rows):,}')


## 2. Define exact retrieval evaluation

Each question is compared with every distinct answer. Metrics are Recall@1, Recall@5, and MRR@10.

In [ ]:
def l2_normalize(embeddings):
    embeddings = np.asarray(embeddings, dtype=np.float32)
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    if np.any(norms == 0):
        raise ValueError('The model produced a zero-length embedding.')
    return embeddings / norms

def calculate_retrieval_results(query_embeddings, corpus_embeddings):
    queries, corpus = l2_normalize(query_embeddings), l2_normalize(corpus_embeddings)
    ranks = np.empty(len(queries), dtype=np.int64)
    top_k = min(RETRIEVED_TOP_K, len(corpus))
    top_indices = np.empty((len(queries), top_k), dtype=np.int64)
    top_scores = np.empty((len(queries), top_k), dtype=np.float32)
    corpus_t = np.ascontiguousarray(corpus.T)
    for start in range(0, len(queries), SEARCH_BATCH_SIZE):
        end = min(start + SEARCH_BATCH_SIZE, len(queries))
        scores = queries[start:end] @ corpus_t
        batch_indices = np.arange(end - start)
        positive_scores = scores[batch_indices, relevant_document_indices[start:end]]
        ranks[start:end] = (scores > positive_scores[:, None]).sum(axis=1) + 1
        candidates = np.argpartition(-scores, top_k - 1, axis=1)[:, :top_k]
        candidate_scores = np.take_along_axis(scores, candidates, axis=1)
        order = np.argsort(-candidate_scores, axis=1, kind='stable')
        top_indices[start:end] = np.take_along_axis(candidates, order, axis=1)
        top_scores[start:end] = np.take_along_axis(candidate_scores, order, axis=1)
    return ranks, top_indices, top_scores

def release_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        if hasattr(torch.cuda, 'ipc_collect'):
            torch.cuda.ipc_collect()

def encode_bge(model_name_or_path, batch_size):
    model = None
    try:
        model = BGEM3FlagModel(str(model_name_or_path), use_fp16=USE_FP16, pooling_method='cls', devices=EVALUATION_DEVICES)
        queries = model.encode_queries(test_queries, batch_size=batch_size, return_dense=True, return_sparse=False, return_colbert_vecs=False)['dense_vecs']
        corpus = model.encode_corpus(corpus_documents, batch_size=batch_size, return_dense=True, return_sparse=False, return_colbert_vecs=False)['dense_vecs']
        return queries, corpus
    finally:
        del model
        release_gpu_memory()

def encode_sentence_transformer(model_name_or_path, batch_size, query_kwargs, corpus_kwargs, trust_remote_code=False):
    model = None
    try:
        model = SentenceTransformer(model_name_or_path, trust_remote_code=trust_remote_code, device=EVALUATION_DEVICE)
        queries = model.encode(test_queries, batch_size=batch_size, convert_to_numpy=True, show_progress_bar=True, **query_kwargs)
        corpus = model.encode(corpus_documents, batch_size=batch_size, convert_to_numpy=True, show_progress_bar=True, **corpus_kwargs)
        return queries, corpus
    finally:
        del model
        release_gpu_memory()

def evaluate_model(spec):
    if spec['family'] == 'bge':
        query_embeddings, corpus_embeddings = encode_bge(spec['model'], spec['batch_size'])
    elif spec['family'] == 'jina_v3':
        query_embeddings, corpus_embeddings = encode_sentence_transformer(spec['model'], spec['batch_size'], {'task': 'retrieval.query', 'prompt_name': 'retrieval.query'}, {'task': 'retrieval.passage', 'prompt_name': 'retrieval.passage'}, True)
    elif spec['family'] == 'snowflake_arctic':
        query_embeddings, corpus_embeddings = encode_sentence_transformer(spec['model'], spec['batch_size'], {'prompt_name': 'query'}, {})
    else:
        raise ValueError(f'Unsupported model family: {spec["family"]}')
    try:
        ranks, top_indices, top_scores = calculate_retrieval_results(query_embeddings, corpus_embeddings)
        return {'label': spec['label'], 'model': str(spec['model']), 'settings': spec['settings'], 'correct_ranks': ranks, 'top_indices': top_indices, 'top_scores': top_scores, 'metrics': {'recall_at_1': float(np.mean(ranks <= 1)), 'recall_at_5': float(np.mean(ranks <= 5)), 'mrr_at_10': float(np.mean(np.where(ranks <= 10, 1.0 / ranks, 0.0)) )}}
    finally:
        del query_embeddings, corpus_embeddings
        release_gpu_memory()


## 3. Evaluate the four retrieval models

In [ ]:
if isinstance(FINE_TUNED_MODEL_PATH, Path) and not FINE_TUNED_MODEL_PATH.is_dir():
    raise FileNotFoundError(f'Fine-tuned model not found: {FINE_TUNED_MODEL_PATH}. Set FINE_TUNED_MODEL_PATH in Setup.')

models_to_evaluate = [
    {'label': 'base_bge_m3', 'model': BASE_MODEL_NAME, 'family': 'bge', 'batch_size': BGE_ENCODE_BATCH_SIZE, 'settings': {'encoder': 'BGEM3FlagModel', 'pooling': 'cls', 'query_mode': 'encode_queries', 'corpus_mode': 'encode_corpus'}},
    {'label': 'fine_tuned_bge_m3', 'model': FINE_TUNED_MODEL_PATH, 'family': 'bge', 'batch_size': BGE_ENCODE_BATCH_SIZE, 'settings': {'encoder': 'BGEM3FlagModel', 'pooling': 'cls', 'query_mode': 'encode_queries', 'corpus_mode': 'encode_corpus'}},
    {'label': 'jina_embeddings_v3', 'model': JINA_MODEL_NAME, 'family': 'jina_v3', 'batch_size': COMPARISON_ENCODE_BATCH_SIZE, 'settings': {'encoder': 'SentenceTransformer', 'query_task': 'retrieval.query', 'corpus_task': 'retrieval.passage'}},
    {'label': 'snowflake_arctic_embed_l_v2', 'model': SNOWFLAKE_MODEL_NAME, 'family': 'snowflake_arctic', 'batch_size': COMPARISON_ENCODE_BATCH_SIZE, 'settings': {'encoder': 'SentenceTransformer', 'query_prompt_name': 'query', 'corpus_prompt_name': None}},
]

evaluation_results = []
for spec in models_to_evaluate:
    print(f"Evaluating {spec['label']}: {spec['model']} (batch size {spec['batch_size']})")
    result = evaluate_model(spec)
    evaluation_results.append(result)
    print(f"{result['label']} — Recall@1: {result['metrics']['recall_at_1']:.2%}; Recall@5: {result['metrics']['recall_at_5']:.2%}; MRR@10: {result['metrics']['mrr_at_10']:.2%}")


## 4. Save comparable metrics and per-query ranks

For each model, save metrics JSON, per-query ranks CSV, retrieved top-5 JSONL, a comparison CSV, and a self-contained HTML report.

In [ ]:
REPORT_DIR.mkdir(parents=True, exist_ok=True)
comparison_rows = []
for result, spec in zip(evaluation_results, models_to_evaluate):
    label = re.sub(r'[^A-Za-z0-9._-]+', '-', result['label']).strip('-_')
    report = {'model': result['model'], 'test_file': str(TEST_INPUT_PATH.resolve()), 'query_count': len(test_rows), 'corpus_document_count': len(corpus_documents), 'metrics': result['metrics'], 'settings': {**result['settings'], 'devices': EVALUATION_DEVICES, 'fp16': USE_FP16, 'encode_batch_size': spec['batch_size'], 'search_batch_size': SEARCH_BATCH_SIZE}}
    (REPORT_DIR / f'metrics_{label}.json').write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    with (REPORT_DIR / f'per_query_ranks_{label}.csv').open('w', encoding='utf-8-sig', newline='') as target:
        writer = csv.DictWriter(target, fieldnames=['row_id', 'source_id', 'question', 'correct_answer_rank', 'recall_at_1', 'recall_at_5', 'reciprocal_rank_at_10'])
        writer.writeheader()
        writer.writerows({'row_id': row_id, 'source_id': source_id, 'question': question, 'correct_answer_rank': int(rank), 'recall_at_1': int(rank <= 1), 'recall_at_5': int(rank <= 5), 'reciprocal_rank_at_10': float(1 / rank) if rank <= 10 else 0.0} for row_id, source_id, question, rank in zip(test_row_ids, test_source_ids, test_queries, result['correct_ranks']))
    with (REPORT_DIR / f'per_query_top_{RETRIEVED_TOP_K}_{label}.jsonl').open('w', encoding='utf-8') as target:
        for row_id, source_id, question, rank, indices, scores in zip(test_row_ids, test_source_ids, test_queries, result['correct_ranks'], result['top_indices'], result['top_scores']):
            retrieved = [{'rank': position, 'answer_row_id': test_row_ids[int(index)], 'answer_source_id': test_source_ids[int(index)], 'score': float(score)} for position, (index, score) in enumerate(zip(indices, scores), 1)]
            target.write(json.dumps({'row_id': row_id, 'source_id': source_id, 'question': question, 'correct_answer_row_id': row_id, 'correct_answer_rank': int(rank), 'retrieved': retrieved}, ensure_ascii=False) + '\n')
    comparison_rows.append({'label': result['label'], 'model': result['model'], **result['metrics']})

with (REPORT_DIR / 'metrics_comparison.csv').open('w', encoding='utf-8-sig', newline='') as target:
    writer = csv.DictWriter(target, fieldnames=['label', 'model', 'recall_at_1', 'recall_at_5', 'mrr_at_10'])
    writer.writeheader(); writer.writerows(comparison_rows)

rows_html = ''.join(f"<tr><td>{html.escape(row['label'])}</td><td>{html.escape(row['model'])}</td><td>{row['recall_at_1']:.2%}</td><td>{row['recall_at_5']:.2%}</td><td>{row['mrr_at_10']:.2%}</td></tr>" for row in comparison_rows)
report_html = f"""<!doctype html><html><head><meta charset='utf-8'><title>dataset1 retrieval evaluation</title><style>body{{font:15px/1.5 Segoe UI,Arial,sans-serif;margin:40px;color:#18212b}}table{{border-collapse:collapse;width:100%}}th,td{{padding:10px;border-bottom:1px solid #d7dee6;text-align:left}}th{{color:#087f73}}h1{{border-bottom:4px solid #087f73;padding-bottom:12px}}</style></head><body><h1>dataset1 Model Evaluation</h1><p>Generated: {datetime.now().astimezone():%Y-%m-%d %H:%M:%S %Z}<br>Queries / corpus: {len(test_rows):,} / {len(corpus_documents):,}<br>Device: {html.escape(EVALUATION_DEVICE)}</p><table><thead><tr><th>Label</th><th>Model</th><th>Recall@1</th><th>Recall@5</th><th>MRR@10</th></tr></thead><tbody>{rows_html}</tbody></table></body></html>"""
(REPORT_DIR / 'evaluation_report.html').write_text(report_html, encoding='utf-8')
for row in comparison_rows:
    print(f"{row['label']}: Recall@1={row['recall_at_1']:.2%}, Recall@5={row['recall_at_5']:.2%}, MRR@10={row['mrr_at_10']:.2%}")
print(f"Comparison: {REPORT_DIR / 'metrics_comparison.csv'}")
print(f"HTML report: {REPORT_DIR / 'evaluation_report.html'}")
